# OCR Processing

Detects scanned (image-only) PDFs in the raw PDF collection and applies OCR to make them text-searchable using Tesseract + Ghostscript via `ocrmypdf`. Text-based PDFs are left untouched.

**Input:** `pdfs/` — raw downloaded PDFs  
**Output:**
- `pdfs_with_ocr/` — OCR-processed versions of scanned PDFs
- `pdfs_final/` — merged collection (OCR versions replace their originals; text-based PDFs copied as-is)

**Prerequisites:** Install system dependencies before running:
- Linux: `sudo apt install tesseract-ocr ghostscript`
- Mac: `brew install tesseract ghostscript`
- Windows: download from [tesseract-ocr.github.io](https://tesseract-ocr.github.io) and [ghostscript.com](https://www.ghostscript.com)

In [ ]:
import shutil
import subprocess
from pathlib import Path

import fitz
import ocrmypdf

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PDFS_DIR  = PROJECT_ROOT / "pdfs"          # raw input PDFs
OCR_DIR   = PROJECT_ROOT / "pdfs_with_ocr" # OCR-processed scanned PDFs
FINAL_DIR = PROJECT_ROOT / "pdfs_final"    # merged final collection

MIN_TEXT_THRESHOLD = 50  # characters per PDF; below this it is treated as scanned

In [ ]:
# Verify that required system tools are available before running
for tool in ["tesseract", "gs"]:
    try:
        subprocess.run([tool, "--version"], capture_output=True, check=True)
        print(f"OK  {tool}")
    except FileNotFoundError:
        print(f"MISSING  {tool} — install it before running (see Prerequisites above)")

In [ ]:
def is_pdf_scanned(pdf_path: Path, min_text_threshold: int = MIN_TEXT_THRESHOLD) -> bool:
    """
    Check if a PDF is scanned (image-only) by counting extracted text characters.
    Only the first 3 pages are checked for efficiency.

    Returns True if scanned, False if text-based.
    """
    try:
        doc = fitz.open(pdf_path)
        total_text = ""

        pages_to_check = min(3, len(doc))
        for page_num in range(pages_to_check):
            total_text += doc.load_page(page_num).get_text()

        doc.close()
        return len(total_text.strip()) < min_text_threshold
    except Exception as e:
        print(f"Error checking {pdf_path.name}: {e}")
        return False


def apply_ocr_to_scanned_pdfs(input_dir: Path, output_dir: Path) -> dict:
    """
    Process PDFs in input_dir, applying OCR only to scanned documents.

    Args:
        input_dir: Folder containing raw PDF files.
        output_dir: Folder where OCR-processed PDFs will be saved.

    Returns:
        Dictionary with processing statistics (total, scanned, text_based, success, failed).
    """
    output_dir.mkdir(exist_ok=True, parents=True)
    stats = {"total": 0, "scanned": 0, "text_based": 0, "success": 0, "failed": 0}

    pdf_files = list(input_dir.glob("*.pdf"))
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return stats

    print(f"Found {len(pdf_files)} PDF files\n")

    for pdf_file in pdf_files:
        stats["total"] += 1
        print(f"[{stats['total']}/{len(pdf_files)}] {pdf_file.name}")

        if is_pdf_scanned(pdf_file):
            stats["scanned"] += 1
            output_file = output_dir / pdf_file.name
            try:
                ocrmypdf.ocr(
                    pdf_file,
                    output_file,
                    skip_text=True,     # skip pages that already have text
                    deskew=True,        # straighten skewed scans
                    rotate_pages=True,  # auto-rotate misoriented pages
                    language="eng",
                    force_ocr=False,
                    progress_bar=False,
                )
                stats["success"] += 1
                print(f"  OCR applied -> {output_file.name}")
            except Exception as e:
                stats["failed"] += 1
                print(f"  Failed: {e}")
        else:
            stats["text_based"] += 1
            print("  Skipped (already has text)")

    print("\n" + "=" * 50)
    print(f"Total:        {stats['total']}")
    print(f"OCR applied:  {stats['success']}")
    print(f"Text-based:   {stats['text_based']}")
    print(f"Failed:       {stats['failed']}")
    print("=" * 50)
    return stats


def create_combined_pdf_folder(original_dir: Path, ocr_dir: Path, output_dir: Path) -> dict:
    """
    Merge original and OCR-processed PDFs into one folder.
    Where an OCR version exists it replaces the original; otherwise the original is copied.

    Args:
        original_dir: Folder with raw PDFs.
        ocr_dir:      Folder with OCR-processed PDFs.
        output_dir:   Destination folder for the merged collection.

    Returns:
        Dictionary with merge statistics (total, copied_original, replaced_with_ocr).
    """
    output_dir.mkdir(exist_ok=True, parents=True)
    stats = {"total": 0, "copied_original": 0, "replaced_with_ocr": 0}

    ocr_files = {f.name for f in ocr_dir.glob("*.pdf")}

    for pdf_file in original_dir.glob("*.pdf"):
        stats["total"] += 1
        output_file = output_dir / pdf_file.name

        if pdf_file.name in ocr_files:
            shutil.copy2(ocr_dir / pdf_file.name, output_file)
            stats["replaced_with_ocr"] += 1
        else:
            shutil.copy2(pdf_file, output_file)
            stats["copied_original"] += 1

    print("=" * 50)
    print(f"Total:         {stats['total']}")
    print(f"OCR versions:  {stats['replaced_with_ocr']}")
    print(f"Originals:     {stats['copied_original']}")
    print("=" * 50)
    return stats

In [ ]:
print("Step 1: Applying OCR to scanned PDFs\n")
ocr_stats = apply_ocr_to_scanned_pdfs(PDFS_DIR, OCR_DIR)
print(f"\nOCR done: {ocr_stats['success']} files processed, saved to {OCR_DIR}")

In [ ]:
print("Step 2: Merging into final PDF folder\n")
merge_stats = create_combined_pdf_folder(PDFS_DIR, OCR_DIR, FINAL_DIR)
print(f"\nMerge done: {merge_stats['total']} PDFs in {FINAL_DIR}")